In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[1]
DATASET_DIR = PROJECT_ROOT/"Dataset"
MODEL_DIR = PROJECT_ROOT/"models"
IR_DIR = PROJECT_ROOT/"Pz and Sz"
FIG_DIR = PROJECT_ROOT/"figs"
print(PROJECT_ROOT)
print(DATASET_DIR)
print(MODEL_DIR)
print(FIG_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)
# Folder_name_list_of_dataset = ['Synthesized_Dataset/Training_data', 'Synthesized_Dataset/Validate_data', 'Synthesized_Dataset/Testing_data']
# Folder_name_list_of_dataset = [DATASET_DIR/i for i in Folder_name_list_of_dataset]
# File_sheet = "Hard_Index.csv"
# MODEL_PTH = MODEL_DIR/"M6_res_Synthetic.pth"

/home/rkobayashi/anc/GFANC-custom
/home/rkobayashi/anc/GFANC-custom/Dataset
/home/rkobayashi/anc/GFANC-custom/models
/home/rkobayashi/anc/GFANC-custom/figs


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from common.loading_real_wave_noise import loading_real_wave_noise
from common.Reading_path_test import loading_paths_from_MAT
from data.Disturbance_generation import Disturbance_generation_from_real_noise
from fxlms.FxLMS_algorithm_v2 import FxLMS, train_fxlms_algorithm, Noisy_FxLMS, train_fxlms_algorithm_for_noisy_ref
from gfanc.Control_filter_selection_pre_now import Control_filter_selection_pre_now
from gfanc.Fixed_filter_noise_cancellation_subfilters import My_Fixed_filter_controller

print(torch.cuda.is_available())

True


In [3]:
# Loading real noises
mdict = {}
fs = 16000
StepSize = 0.0001
sound_name = 'Helicopter'
waveform, resample_rate = loading_real_wave_noise(folde_name=PROJECT_ROOT/'Real Noise Examples/', sound_name=sound_name+'.wav')
waveform = waveform[0].unsqueeze(0)
dtype = torch.float32

In [4]:

# Loading path
subfolder = "Dongyuan"
Pri_path, Secon_path = loading_paths_from_MAT(folder=IR_DIR, subfolder=subfolder, Pri_path_file_name='Primary_path.mat', Sec_path_file_name='Secondary_path.mat')
Secon_path = torch.tensor(Secon_path, dtype=dtype)
Dis, Fx, Re = Disturbance_generation_from_real_noise(fs=fs, Repet=0, wave_form=waveform, Pri_path=Pri_path, Sec_path=Secon_path)
# Dis: disturbance (cotrolled noise)， Fx: fixed-x signal, Re: repeated waveform (primary_noise) Repetition=Repet+1
import matplotlib as mpl

mpl.rcParams['agg.path.chunksize'] = 10000

In [8]:
def additional_noise(signal, snr_db):
    signal_power = signal.square().mean()
    additional_noise = torch.randn_like(signal, dtype=signal_power.dtype, device=signal_power.device)
    snr = 10 ** (snr_db / 10)
    noise_power = signal_power / snr
    # print(signal_power)
    # print(noise_power)
    return signal + noise_power.sqrt() * additional_noise

In [9]:
# Noisy_FxLMS
import math
Ref = Re
filter_len=1024
StepSizes = [ StepSize * math.sqrt(10) ** i for i in range(4)]
snrs = [20, 10, 5, 2.5, 1, 0, -1]
for snr in snrs:
    Ref = additional_noise(Re, snr)
    for stepsize in StepSizes:
        controller = Noisy_FxLMS(Len=filter_len) # 1024 is the same size of coeffient vector of fixed-filter
        ErrorFxLMS = train_fxlms_algorithm_for_noisy_ref(controller, Ref, Dis, Secon_path, filter_len, stepsize)
        print(10*torch.log10(torch.tensor(ErrorFxLMS, dtype=dtype).square().sum()/Dis.square().sum()))


[========================================================================] 100%


tensor(-4.9908)


[========================================================================] 100%


tensor(-7.5684)


[========================================================================] 100%


...

tensor(-1.4095)


[========================================================================] 100%


tensor(124.0570)


tensor(inf)


[========================================================================] 100%


In [9]:
ErrorFixeds = {}
ErrorFxLMSs = {}

In [22]:
# FxLMS
StepSizes = [ StepSize * 10 ** i for i in range(5)]
for stepsize in StepSizes:
    controller = FxLMS(Len=1024) # 1024 is the same size of coeffient vector of fixed-filter
    ErrorFxLMS = train_fxlms_algorithm(Model=controller, Ref=Fx, Disturbance=Dis, Stepsize=stepsize) # torch.Size([320000])
    ErrorFxLMSs[stepsize] = 10*torch.log10(torch.tensor(ErrorFxLMS).square().sum()/Dis.square().sum())
    print(ErrorFxLMSs[stepsize])


[========================================================================] 100%


tensor(-5.1006)


[========================================================================] 100%


tensor(-11.0309)


[========================================================================] 100%


tensor(-17.7002)


[========================================================================] 100%


tensor(-23.3256)


tensor(nan)


[========================================================================] 100%


In [23]:
print(ErrorFxLMSs)

{0.0001: tensor(-5.1006), 0.001: tensor(-11.0309), 0.01: tensor(-17.7002), 0.1: tensor(-23.3256), 1.0: tensor(nan)}


In [ ]:
# Noisy_FxLMS
Ref = Re
filter_len=1024
StepSizes = [ StepSize * 10 ** i for i in range(5)]
for stepsize in StepSizes:
    controller = Noisy_FxLMS(Len=filter_len) # 1024 is the same size of coeffient vector of fixed-filter
    ErrorFxLMS = train_fxlms_algorithm_for_noisy_ref(controller, Ref, Dis, Secon_path, filter_len, stepsize)
    print(10*torch.log10(torch.tensor(ErrorFxLMS, dtype=dtype).square().sum()/Dis.square().sum()))


[========================================================================] 100%


tensor(-5.0345)


[========================================================================] 100%


tensor(-10.7908)


[========================================================================] 100%


tensor(nan)


[========================================================================] 100%


tensor(nan)


tensor(nan)


[========================================================================] 100%
